# Script 2 — Preparação & Engenharia de Features
**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

Este notebook é o **segundo passo do pipeline**. Ele recebe o dataset consolidado produzido
pelo Script 1 (`dataset_cvm_consolidado.parquet`) e o transforma em artefatos prontos para
a modelagem de ML do Script 3.

**Etapas executadas:**
1. Carregamento e validação do dataset consolidado
2. Seleção dos exercícios anuais (DFP) — com prioridade explícita sobre ITR
3. Remoção de colunas com excesso de nulos (limiar configurável)
4. Imputação por mediana do setor com fallback global
5. Winsorização de outliers extremos por IQR × fator por setor
6. Engenharia de features: crescimento YoY com verificação de continuidade temporal
7. Codificação one-hot do setor
8. Criação dos targets por shift temporal com validação de gap
9. Seleção de features por correlação de Pearson + RFE com Ridge
10. Split temporal treino/teste preservando ordem cronológica por empresa
11. Persistência de artefatos para o Script 3

## Etapa 0 — Dependências, logging e configuração global

**O que faz:**
Importa bibliotecas, configura logging estruturado (espelhando o padrão do Script 1)
e define todos os parâmetros configuráveis do pipeline em um único bloco.
Centralizar os parâmetros aqui facilita ajustes sem precisar varrer o notebook.

In [ ]:
import logging
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_selection import RFE
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', 40)

# ── Logging estruturado (mesmo padrão do Script 1) ────────────────────────
logger = logging.getLogger('pipeline_preparacao')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()

PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)

_fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s',
                         datefmt='%Y-%m-%d %H:%M:%S')
_sh = logging.StreamHandler()
_sh.setLevel(logging.INFO)
_sh.setFormatter(_fmt)
logger.addHandler(_sh)

_fh = logging.FileHandler(PASTA_SAIDA / 'pipeline_preparacao.log', mode='w', encoding='utf-8')
_fh.setLevel(logging.DEBUG)
_fh.setFormatter(_fmt)
logger.addHandler(_fh)

# ── Parâmetros configuráveis ──────────────────────────────────────────────
# Todos os hiperparâmetros do pipeline de preparação ficam aqui.
# Alterar apenas este bloco para ajustar o comportamento global.

# Lista completa de KPIs (sincronizada com Script 1 — 19 indicadores)
LISTA_KPIS = [
    'margem_bruta', 'margem_ebit', 'margem_liquida', 'margem_ebitda',
    'roe', 'roa', 'liquidez_corrente', 'liquidez_imediata',
    'endividamento', 'alavancagem_de', 'div_liquida', 'cobertura_juros',
    'giro_ativo', 'fco_receita', 'fco_lucro', 'EBITDA',
    'FCF', 'margem_fcf', 'conversao_caixa',   # ← 3 KPIs de caixa adicionados no V5
]

# Targets de predição — o que os modelos do Script 3 devem prever
TARGET_COLS = {
    'TARGET_DRE_3.01': 'DRE_3.01',  # Receita Líquida no exercício t+1
    'TARGET_DRE_3.11': 'DRE_3.11',  # Lucro Líquido no exercício t+1
    'TARGET_EBITDA':   'EBITDA',    # EBITDA no exercício t+1
}

# Parâmetros de limpeza e feature engineering
LIMIAR_NULO     = 0.80   # colunas com >80% nulos são removidas
FATOR_WINSOR    = 3.0    # clip em Q1 - 3×IQR e Q3 + 3×IQR (por setor)
MAX_COLS_YOY    = 16     # limite de variáveis YoY (evita explosão de features)
CLIP_YOY        = 5.0    # limita variação YoY a ±500% (remove distorções)
CORR_MIN        = 0.10   # correlação mínima de Pearson para pré-seleção
N_FEATURES_RFE  = 15     # número de features finais via RFE
FRAC_TREINO     = 0.75   # proporção do período mais antigo usada para treino

logger.info("Script 2 iniciado | Parâmetros: limiar_nulo=%.0f%% | winsor=%.1f | "
            "yoy_max=%d | corr_min=%.2f | rfe_n=%d | frac_treino=%.0f%%",
            LIMIAR_NULO*100, FATOR_WINSOR, MAX_COLS_YOY,
            CORR_MIN, N_FEATURES_RFE, FRAC_TREINO*100)


## Etapa 1 — Carregamento e validação do dataset consolidado

**O que faz:**
Lê o Parquet produzido pelo Script 1 e executa 4 verificações de integridade:
existência das colunas obrigatórias, 25 empresas âncora, cobertura mínima
dos KPIs e validade da coluna `DT_REFER`.

Se o dataset não passar nas verificações críticas, o pipeline para imediatamente
com uma mensagem clara — sem continuar silenciosamente com dados inválidos.

In [ ]:
# ── Carregamento ──────────────────────────────────────────────────────────
cam_parquet = PASTA_SAIDA / 'dataset_cvm_consolidado.parquet'
if not cam_parquet.exists():
    raise FileNotFoundError(
        f"Parquet não encontrado: {cam_parquet}\n"
        "Execute o Script 1 (01_cvm_processamento_V5.ipynb) antes de continuar."
    )

dataset = pd.read_parquet(cam_parquet)
logger.info("Dataset carregado: %d × %d | %s",
            *dataset.shape, cam_parquet.name)

# ── Validação 1: DT_REFER deve ser datetime ───────────────────────────────
if 'DT_REFER' in dataset.columns:
    dataset['DT_REFER'] = pd.to_datetime(dataset['DT_REFER'], errors='coerce', utc=True)
    dataset['ANO'] = dataset['DT_REFER'].dt.year.astype('Int64')
    logger.info("DT_REFER convertida | ANO extraído | range: %s → %s",
                dataset['DT_REFER'].min().date(), dataset['DT_REFER'].max().date())

# ── Validação 2: colunas obrigatórias ────────────────────────────────────
COLS_OBRIGATORIAS = ['CNPJ_CIA', 'NOME_CIA', 'SETOR', 'ANO', 'ORIGEM', 'DT_REFER']
faltando = [c for c in COLS_OBRIGATORIAS if c not in dataset.columns]
if faltando:
    raise ValueError(f"Colunas obrigatórias ausentes: {faltando}")

# ── Validação 3: empresas âncora ─────────────────────────────────────────
n_emp = dataset['NOME_CIA'].nunique()
if n_emp < 25:
    logger.warning("Apenas %d/25 empresas no dataset — verifique o Script 1", n_emp)
else:
    logger.info("Empresas: %d/25 ✅", n_emp)

# ── Validação 4: cobertura dos KPIs ──────────────────────────────────────
kpis_presentes = [k for k in LISTA_KPIS if k in dataset.columns]
kpis_ausentes  = [k for k in LISTA_KPIS if k not in dataset.columns]
logger.info("KPIs presentes: %d/%d", len(kpis_presentes), len(LISTA_KPIS))
if kpis_ausentes:
    logger.warning("KPIs não encontrados no dataset: %s", kpis_ausentes)

for kpi in kpis_presentes:
    cob = dataset[kpi].notna().mean()
    nivel = '✅' if cob >= 0.9 else ('⚠️' if cob >= 0.5 else '❌')
    logger.info("  %s %-25s %.0f%%", nivel, kpi, cob * 100)

# Resumo
print(f"\n{'='*60}")
print(f"  Dataset consolidado carregado com sucesso")
print(f"  Shape   : {dataset.shape[0]:,} linhas × {dataset.shape[1]} colunas")
print(f"  Empresas: {n_emp} / 25")
print(f"  Origens : {dataset['ORIGEM'].value_counts().to_dict()}")
print(f"  Anos    : {sorted(dataset['ANO'].dropna().astype(int).unique())}")
print(f"  KPIs    : {len(kpis_presentes)} / {len(LISTA_KPIS)} presentes")
print(f"{'='*60}")


## Etapa 2 — Seleção dos exercícios anuais (DFP)

**O que faz:**
Filtra o dataset para manter **um único registro por empresa × ano** correspondente
ao exercício anual (DFP).

**Por que não basta `drop_duplicates`:**
A versão original do Script 2 fazia `drop_duplicates(['CNPJ_CIA','ANO'])` com critério
de máximo de KPIs não-nulos. O problema é que o ITR do Q4 (dezembro) geralmente tem
mais KPIs preenchidos do que o DFP anual correspondente — então o dedup escolhia o ITR
em 24 das 25 empresas. Modelos treinados em ITR trimestrais de dez/ano têm vazamento
de dados e não generalizam corretamente para predição anual.

**Estratégia correta:**
1. Filtrar `ORIGEM == 'DFP'` primeiro, garantindo que apenas demonstrações anuais completas entrem
2. Para a Raízen (exercício fiscal abril→março), o DFP cai em março — isso é tratado
   corretamente pois a coluna `ORIGEM` identifica o tipo de documento, não o mês
3. Dentro dos DFPs, `drop_duplicates` por `(CNPJ_CIA, ANO)` mantém a retificação mais recente

In [ ]:
# ── Filtro: apenas DFPs ──────────────────────────────────────────────────
# A distinção DFP/ITR é feita pela coluna ORIGEM, não pelo mês da DT_REFER.
# Isso trata corretamente empresas com exercício fiscal atípico (ex: Raízen: abr→mar).

n_total = len(dataset)
dfp = dataset[dataset['ORIGEM'] == 'DFP'].copy()
n_dfp = len(dfp)
logger.info("Filtro DFP: %d → %d linhas (descartados %d ITR)",
            n_total, n_dfp, n_total - n_dfp)

if dfp.empty:
    raise ValueError(
        "Nenhum registro DFP encontrado. "
        "Verifique se o Script 1 processou os ZIPs de DFP corretamente."
    )

# ── Deduplicação intra-DFP por empresa × ano ─────────────────────────────
# Dentro dos DFPs, pode haver retificações (versões 1 e 2 do mesmo exercício).
# Mantemos o registro com maior completude de KPIs — proxy da retificação mais recente.
dfp['_n_kpis'] = dfp[kpis_presentes].notna().sum(axis=1)

n_ant = len(dfp)
dfp = (dfp
    .sort_values(['CNPJ_CIA', 'ANO', '_n_kpis'], ascending=[True, True, False])
    .drop_duplicates(subset=['CNPJ_CIA', 'ANO'], keep='first')
    .drop(columns=['_n_kpis'])
    .sort_values(['CNPJ_CIA', 'ANO'])
    .reset_index(drop=True)
)
rem = n_ant - len(dfp)
if rem:
    logger.info("Dedup DFP: %d → %d (-%d retificações)", n_ant, len(dfp), rem)

# ── Validação: cobertura por empresa × ano ───────────────────────────────
cobertura = (dfp.groupby('NOME_CIA')
               .agg(Anos=('ANO', lambda x: sorted(x.astype(int).unique())),
                    N=('ANO', 'count'))
               .sort_values('N', ascending=False))
logger.info("Cobertura DFP por empresa:\n%s", cobertura.to_string())

# Alerta para empresas com menos de 4 exercícios
poucas = cobertura[cobertura['N'] < 4]
if not poucas.empty:
    logger.warning("Empresas com < 4 exercícios DFP: %s", poucas.index.tolist())

print(f"\nDataset após seleção DFP: {dfp.shape[0]} linhas × {dfp.shape[1]} colunas")
print(f"Empresas: {dfp['NOME_CIA'].nunique()} | Anos: {sorted(dfp['ANO'].dropna().astype(int).unique())}")
print(f"\nDistribuição por empresa:")
print(cobertura.to_string())


## Etapa 3 — Remoção de colunas com excesso de nulos

**O que faz:**
Remove colunas numéricas com mais de `LIMIAR_NULO` (80%) de valores ausentes.

**Por que é necessário:**
O dataset consolidado tem 736 colunas — a maioria são contas contábeis brutas
(DRE_3.xx, BPA_1.xx, etc.) que existem para algumas empresas mas não para outras.
Colunas com >80% de nulos não têm massa crítica para imputação confiável e apenas
aumentam ruído e custo computacional nos modelos.

**O que é preservado:**
Colunas de metadados (CNPJ, NOME_CIA, SETOR, etc.) e todos os KPIs calculados
são preservados independentemente da taxa de nulos — eles passarão pela imputação.

In [ ]:
# ── Remoção de colunas esparças ──────────────────────────────────────────
cols_num = dfp.select_dtypes(include='number').columns.tolist()

# Remove KPIs e colunas-chave da análise — nunca serão excluídos por nulos
COLS_PROTEGIDAS = set(kpis_presentes + ['ANO'])
cols_candidatas = [c for c in cols_num if c not in COLS_PROTEGIDAS]

taxa_nulo = dfp[cols_candidatas].isnull().mean()
cols_excluir = taxa_nulo[taxa_nulo > LIMIAR_NULO].index.tolist()

# Log detalhado de quais grupos são mais afetados
grupos_exc = {}
for c in cols_excluir:
    prefixo = c.split('_')[0] if '_' in c else 'outro'
    grupos_exc[prefixo] = grupos_exc.get(prefixo, 0) + 1
logger.info("Colunas excluídas (>%.0f%% nulos): %d | por grupo: %s",
            LIMIAR_NULO * 100, len(cols_excluir), grupos_exc)

dfp = dfp.drop(columns=cols_excluir)

# Atualiza lista de KPIs (devem estar todos ainda presentes)
kpis_presentes = [k for k in kpis_presentes if k in dfp.columns]
kpis_ausentes_pos = [k for k in LISTA_KPIS if k not in dfp.columns]
if kpis_ausentes_pos:
    logger.warning("KPIs removidos pelo filtro de nulos (inesperado): %s",
                   kpis_ausentes_pos)

print(f"Colunas removidas (>{LIMIAR_NULO:.0%} nulos): {len(cols_excluir)}")
print(f"Dataset após remoção: {dfp.shape[0]} linhas × {dfp.shape[1]} colunas")
print(f"KPIs mantidos: {len(kpis_presentes)}/{len(LISTA_KPIS)}")


## Etapa 4 — Imputação por mediana do setor

**O que faz:**
Imputa valores ausentes nos KPIs usando a mediana do setor no mesmo período.
Se o setor inteiro for nulo para aquele KPI no período, aplica a mediana global.

**Lógica:**
Usar mediana do setor preserva heterogeneidade entre setores — a margem mediana
do setor de Petróleo é muito diferente da do Varejo, e imputar pela global
introduziria viés sistemático. A mediana (em vez da média) é robusta a outliers
que ainda existam antes da winsorização.

In [ ]:
# ── Imputação em cascata: mediana setor → mediana global ─────────────────
taxa_nulo_pre = dfp[kpis_presentes].isnull().mean()
n_nulos_pre   = dfp[kpis_presentes].isnull().sum().sum()

for kpi in kpis_presentes:
    n_nulo = dfp[kpi].isnull().sum()
    if n_nulo == 0:
        continue

    # Nível 1: mediana do setor
    mediana_setor = dfp.groupby('SETOR')[kpi].transform('median')
    # Nível 2: mediana global (fallback quando setor inteiro é nulo)
    mediana_global = dfp[kpi].median()

    dfp[kpi] = (dfp[kpi]
        .fillna(mediana_setor)
        .fillna(mediana_global)
    )
    n_restante = dfp[kpi].isnull().sum()
    if n_restante:
        logger.warning("Imputação incompleta: %s | %d nulos restantes", kpi, n_restante)
    else:
        logger.debug("Imputação OK: %s | %d nulos → 0", kpi, n_nulo)

n_nulos_pos = dfp[kpis_presentes].isnull().sum().sum()
logger.info("Imputação concluída: %d nulos → %d | redução: %.0f%%",
            n_nulos_pre, n_nulos_pos,
            (1 - n_nulos_pos / max(n_nulos_pre, 1)) * 100)

print(f"Nulos nos KPIs: {n_nulos_pre} → {n_nulos_pos}")
print(f"Taxa restante : {dfp[kpis_presentes].isnull().mean().mean():.2%}")


## Etapa 5 — Winsorização de outliers (IQR × fator por setor)

**O que faz:**
Trunca valores extremos de cada KPI dentro de cada setor usando o critério
`[Q1 − FATOR×IQR, Q3 + FATOR×IQR]`. O fator padrão é 3.0 (mais conservador
que o clássico 1.5 de Tukey), adequado para dados financeiros onde valores
extremos legítimos são comuns.

**Por que por setor:**
Os limites de normalidade variam muito entre setores — uma margem EBITDA de 60%
é normal para Petróleo mas seria um outlier extremo para Varejo. Winsorizações
globais eliminariam observações legítimas de setores com características distintas.

In [ ]:
# ── Winsorização por setor ────────────────────────────────────────────────
def winsorizacao_setor(df: pd.DataFrame, col: str, fator: float = 3.0) -> pd.DataFrame:
    """
    Trunca valores extremos de uma coluna dentro de cada setor.
    Preserva observações — apenas clippa os valores, não remove linhas.
    Retorna o DataFrame com a coluna winsorizadadentro de cada grupo.
    """
    def _clip_grupo(grupo):
        q1 = grupo[col].quantile(0.25)
        q3 = grupo[col].quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            return grupo   # setor homogêneo — sem winsorização
        low  = q1 - fator * iqr
        high = q3 + fator * iqr
        n_clip = ((grupo[col] < low) | (grupo[col] > high)).sum()
        if n_clip:
            logger.debug("Winsor | %-25s | setor %-15s | %d valores clipados",
                         col, grupo.name, n_clip)
        grupo = grupo.copy()
        grupo[col] = grupo[col].clip(lower=low, upper=high)
        return grupo

    return df.groupby('SETOR', group_keys=False).apply(_clip_grupo)

n_total_clipados = 0
for kpi in kpis_presentes:
    antes = dfp[kpi].copy()
    dfp = winsorizacao_setor(dfp, kpi, FATOR_WINSOR)
    n_clip = (dfp[kpi] != antes).sum()
    n_total_clipados += n_clip

logger.info("Winsorização concluída | fator=%.1f | total clipado: %d valores",
            FATOR_WINSOR, n_total_clipados)
print(f"✅ Winsorização aplicada (fator={FATOR_WINSOR}) | {n_total_clipados} valores truncados")
print(f"   Dataset: {dfp.shape}")


## Etapa 6 — Variáveis de crescimento YoY

**O que faz:**
Calcula a variação percentual Year-over-Year (YoY) dos KPIs principais,
criando até `MAX_COLS_YOY` novas features.

**Cuidados implementados:**

**Verificação de continuidade temporal:** `pct_change()` calcula a variação
entre linhas consecutivas do grupo, sem saber se há um gap de anos. Após o cálculo,
qualquer YoY onde o exercício anterior não é exatamente `ANO - 1` é anulado — evita
criar uma variação espúria entre 2021 e 2023 (por exemplo) quando 2022 está ausente.

**Clip de variação extrema:** limita o YoY a ±`CLIP_YOY` (±500%) para remover
distorções causadas por denominadores próximos de zero (ex: `roe_yoy` quando
o PL era quase zero no ano anterior).

In [ ]:
# ── Variáveis YoY com verificação de continuidade temporal ───────────────
dfp = dfp.sort_values(['CNPJ_CIA', 'ANO']).reset_index(drop=True)

# KPIs usados para YoY: exclui métricas absolutas (div_liquida, EBITDA)
# pct_change em valores absolutos de grande escala gera features menos úteis
KPIS_YOY = [k for k in kpis_presentes
             if k not in ('div_liquida', 'EBITDA', 'FCF')][:MAX_COLS_YOY]

cols_yoy = []
for kpi in KPIS_YOY:
    col_yoy = f'{kpi}_yoy'

    # Variação percentual raw
    yoy_raw = (dfp
        .groupby('CNPJ_CIA')[kpi]
        .pct_change()
        .replace([np.inf, -np.inf], np.nan)
        .clip(-CLIP_YOY, CLIP_YOY)
    )

    # Anula onde o exercício anterior não é ANO-1 (gap temporal)
    # Isso evita variações espúrias entre exercícios não consecutivos
    ano_prev = dfp.groupby('CNPJ_CIA')['ANO'].shift(1)
    gap_invalido = (dfp['ANO'] - ano_prev) != 1
    yoy_raw[gap_invalido] = np.nan

    dfp[col_yoy] = yoy_raw
    cols_yoy.append(col_yoy)

# Feature de aceleração do crescimento (segunda derivada da margem EBITDA)
# Captura se o crescimento está acelerando ou desacelerando — útil para ML
if 'margem_ebitda_yoy' in dfp.columns:
    dfp['aceleracao_ebitda'] = (
        dfp.groupby('CNPJ_CIA')['margem_ebitda_yoy']
        .diff()
        .replace([np.inf, -np.inf], np.nan)
    )
    # Anula em gaps temporais
    dfp.loc[gap_invalido, 'aceleracao_ebitda'] = np.nan
    cols_yoy.append('aceleracao_ebitda')

n_nulos_yoy = dfp[cols_yoy].isnull().mean().mean()
logger.info("YoY: %d features criadas | nulos médios: %.0f%% (esperado ~25%% — primeira obs de cada empresa)",
            len(cols_yoy), n_nulos_yoy * 100)
print(f"Features YoY criadas: {len(cols_yoy)}")
print(f"Nulos médios nas YoY: {n_nulos_yoy:.0%} (esperado: ~25% — 1ª obs de cada empresa não tem anterior)")
print(f"Dataset: {dfp.shape}")


## Etapa 7 — Codificação one-hot do setor

**O que faz:**
Transforma a coluna categórica `SETOR` em 5 colunas binárias com prefixo `setor_`.
O `drop_first=False` é intencional: com apenas 5 setores e modelos como Ridge
regularizados, manter todas as categorias melhora a interpretabilidade sem
introduzir multicolinearidade problemática.

In [ ]:
# ── One-hot encoding do setor ─────────────────────────────────────────────
# drop_first=False: com regularização (Ridge/Lasso), manter todas as categorias
# é preferível para interpretabilidade — o modelo aprende o efeito de cada setor.
dfp = pd.get_dummies(dfp, columns=['SETOR'], prefix='setor', dtype=float)
cols_setor = sorted([c for c in dfp.columns if c.startswith('setor_')])

logger.info("One-hot SETOR: %d colunas criadas | %s", len(cols_setor), cols_setor)
print(f"Colunas de setor: {cols_setor}")
print(f"Dataset: {dfp.shape}")


## Etapa 8 — Criação dos targets por shift temporal

**O que faz:**
Cria os três targets de predição como o valor do KPI no **próximo exercício anual**
(deslocamento temporal de 1 ano dentro de cada empresa).

**Conceito de data leakage temporal:**
Se um modelo usar features do ano *t* para prever o ano *t*, não há predição — há
apenas correlação contemporânea. O target correto é o valor em *t+1*, e as features
são os dados disponíveis no final do ano *t* (antes da divulgação de *t+1*).

**Validação de gap:**
O `shift(-1)` do pandas não sabe que entre dois registros consecutivos pode haver
um gap de 2 anos (empresa sem dados em um exercício). Se o próximo ANO não for
`ANO + 1` exato, o target é anulado — isso é correto: não há como prever t+2
a partir de t quando t+1 está ausente.

In [ ]:
# ── Targets: valor do KPI no exercício seguinte ──────────────────────────
dfp = dfp.sort_values(['CNPJ_CIA', 'ANO']).reset_index(drop=True)

targets_criados = []
for target_col, source_col in TARGET_COLS.items():
    if source_col not in dfp.columns:
        logger.warning("Fonte do target '%s' não encontrada: '%s'", target_col, source_col)
        continue

    # Shift temporal dentro de cada empresa
    dfp[target_col] = dfp.groupby('CNPJ_CIA')[source_col].shift(-1)

    # Anula targets onde o próximo exercício não é consecutivo (gap temporal)
    ano_next = dfp.groupby('CNPJ_CIA')['ANO'].shift(-1)
    gap = ano_next - dfp['ANO']
    n_gap = (gap.notna() & (gap != 1)).sum()
    if n_gap:
        logger.warning("Target '%s': %d obs anuladas por gap temporal (ano_next ≠ ANO+1)",
                       target_col, n_gap)
    dfp.loc[gap != 1, target_col] = np.nan

    n_validos = dfp[target_col].notna().sum()
    n_total   = len(dfp)
    targets_criados.append(target_col)
    logger.info("Target %-22s: %d/%d obs válidas (%.0f%%)",
                target_col, n_validos, n_total, n_validos/n_total*100)
    print(f"  {target_col}: {n_validos}/{n_total} obs válidas")

# Remove registros sem nenhum target (última observação de cada empresa, tipicamente)
n_ant = len(dfp)
dfp = dfp[dfp[targets_criados].notna().any(axis=1)].reset_index(drop=True)
n_rem = n_ant - len(dfp)
logger.info("Remoção de linhas sem target: %d → %d (-%d)", n_ant, len(dfp), n_rem)
print(f"\nDataset com targets: {dfp.shape[0]} × {dfp.shape[1]} (removidas {n_rem} sem t+1)")


## Etapa 9 — Seleção de features por correlação e RFE

**O que faz:**
Seleciona as features mais relevantes para o target principal (`TARGET_DRE_3.01`)
em dois passos:

**Pré-seleção por correlação de Pearson:** mantém apenas features com `|r| > CORR_MIN`
(0.10). Isso elimina features completamente irrelevantes antes do RFE — importante
porque com ~100 observações e 50+ features candidatas, o RFE por si só pode ser
instável.

**RFE com Ridge:** seleciona as `N_FEATURES_RFE` mais importantes usando
eliminação recursiva de features com modelo Ridge regularizado. O Ridge é preferível
ao OLS neste contexto porque o dataset pequeno (n≈100) torna o OLS instável
numericamente.

**Nota sobre escopo:** a seleção é feita apenas para o target de Receita Líquida.
Os mesmos features são usados nos três targets no Script 3 — uma seleção de features
por target aumentaria o risco de overfitting dado o tamanho do dataset.

In [ ]:
# ── Pool de features candidatas ──────────────────────────────────────────
FEATURES_CANDIDATAS = (
    kpis_presentes
    + cols_yoy
    + cols_setor
    + (['aceleracao_ebitda'] if 'aceleracao_ebitda' in dfp.columns else [])
)
FEATURES_CANDIDATAS = [f for f in FEATURES_CANDIDATAS if f in dfp.columns]
logger.info("Features candidatas: %d", len(FEATURES_CANDIDATAS))

target_principal = 'TARGET_DRE_3.01'
if target_principal not in dfp.columns:
    logger.error("Target principal '%s' não encontrado — seleção de features impossível",
                 target_principal)
    FEATURES_SELECIONADAS = FEATURES_CANDIDATAS[:N_FEATURES_RFE]
else:
    # ── Passo 1: pré-seleção por correlação de Pearson ───────────────────
    df_sel = dfp[FEATURES_CANDIDATAS + [target_principal]].dropna()
    X_sel  = df_sel[FEATURES_CANDIDATAS]
    y_sel  = df_sel[target_principal]

    if len(df_sel) < 20:
        logger.warning("Apenas %d obs disponíveis para seleção — usando todas as features",
                       len(df_sel))
        FEATURES_SELECIONADAS = FEATURES_CANDIDATAS[:N_FEATURES_RFE]
    else:
        corr_abs = X_sel.corrwith(y_sel).abs().sort_values(ascending=False)
        FEATURES_CORR = corr_abs[corr_abs >= CORR_MIN].index.tolist()
        logger.info("Pré-seleção Pearson (|r| >= %.2f): %d → %d features",
                    CORR_MIN, len(FEATURES_CANDIDATAS), len(FEATURES_CORR))

        # ── Passo 2: RFE com Ridge ────────────────────────────────────────
        n_rfe = min(N_FEATURES_RFE, len(FEATURES_CORR))
        if len(FEATURES_CORR) <= n_rfe:
            FEATURES_SELECIONADAS = FEATURES_CORR
            logger.info("Features pós-correlação (%d) ≤ N_FEATURES_RFE (%d) — RFE não necessário",
                        len(FEATURES_CORR), n_rfe)
        else:
            X_corr   = df_sel[FEATURES_CORR]
            scaler   = StandardScaler()
            X_scaled = scaler.fit_transform(X_corr)

            rfe = RFE(
                estimator=Ridge(alpha=1.0),
                n_features_to_select=n_rfe,
                step=2,
            )
            rfe.fit(X_scaled, y_sel)
            FEATURES_SELECIONADAS = [FEATURES_CORR[i]
                                     for i, sel in enumerate(rfe.support_) if sel]
            logger.info("RFE Ridge: %d → %d features selecionadas",
                        len(FEATURES_CORR), len(FEATURES_SELECIONADAS))

        # Ranking das features selecionadas por correlação
        print(f"\n  Features selecionadas ({len(FEATURES_SELECIONADAS)}):")
        for f in FEATURES_SELECIONADAS:
            r = corr_abs.get(f, 0.0)
            print(f"    {f:<35} |r| = {r:.3f}")

print(f"\nFeatures finais: {len(FEATURES_SELECIONADAS)}")


## Etapa 10 — Split temporal treino/teste (75%/25%)

**O que faz:**
Divide os dados em treino e teste preservando a **ordem cronológica dentro de cada
empresa**. Os 75% de exercícios mais antigos de cada empresa vão para treino; os 25%
mais recentes vão para teste.

**Por que temporal e não aleatório:**
Um split aleatório permitiria que dados futuros "contaminassem" o treino — por exemplo,
features do ano 2024 de uma empresa no treino quando o modelo ainda não "viu" os dados
de 2022 dessa empresa. Em séries temporais financeiras, o split deve sempre respeitar
a ordem cronológica para simular o cenário real de predição.

In [ ]:
# ── Split temporal por empresa ────────────────────────────────────────────
def split_temporal_empresa(grupo: pd.DataFrame, frac_treino: float = 0.75) -> pd.DataFrame:
    """
    Marca os primeiros `frac_treino` × n registros (ordenados por ANO) como treino
    e os restantes como teste. Garante pelo menos 1 observação por split.
    """
    grupo = grupo.sort_values('ANO').copy()
    n = len(grupo)
    n_treino = max(1, round(n * frac_treino))
    n_treino = min(n_treino, n - 1)    # garante pelo menos 1 obs no teste
    grupo['split'] = 'teste'
    grupo.iloc[:n_treino, grupo.columns.get_loc('split')] = 'treino'
    return grupo

dfp = dfp.groupby('CNPJ_CIA', group_keys=False).apply(
    split_temporal_empresa, frac_treino=FRAC_TREINO
)

treino = dfp[dfp['split'] == 'treino'].copy()
teste  = dfp[dfp['split'] == 'teste'].copy()

n_tot    = len(dfp)
n_treino = len(treino)
n_teste  = len(teste)
logger.info("Split temporal: %d treino (%.0f%%) | %d teste (%.0f%%)",
            n_treino, n_treino/n_tot*100, n_teste, n_teste/n_tot*100)

print(f"\nTreino: {n_treino} obs ({n_treino/n_tot:.0%})")
print(f"Teste : {n_teste} obs ({n_teste/n_tot:.0%})")
print(f"\nDistribuição por empresa:")
split_tab = (dfp.groupby(['NOME_CIA', 'split'])['ANO']
               .apply(lambda x: sorted(x.astype(int).unique()))
               .unstack(fill_value=[]))
print(split_tab.to_string())

# Alerta de contaminação: verificar que todos os anos de teste > todos os anos de treino
for emp, grp in dfp.groupby('CNPJ_CIA'):
    anos_treino = set(grp[grp['split']=='treino']['ANO'].dropna().astype(int))
    anos_teste  = set(grp[grp['split']=='teste']['ANO'].dropna().astype(int))
    if anos_treino and anos_teste and max(anos_treino) >= min(anos_teste):
        logger.warning("Possível sobreposição temporal | empresa %s | "
                       "treino max=%d | teste min=%d",
                       emp, max(anos_treino), min(anos_teste))


## Etapa 11 — Persistência dos artefatos para o Script 3

**O que faz:**
Salva todos os artefatos necessários para a modelagem em dois formatos:

**Parquet** (DataFrames): dataset completo, treino e teste — preserva tipos,
timezone e permite leitura eficiente pelo Script 3.

**Pickle** (metadados): listas de features, targets, KPIs e parâmetros — objetos
Python que não têm representação natural em Parquet.

**Relatório JSON**: registra todas as decisões de preparação para rastreabilidade
acadêmica — equivalente ao `auditoria_processamento.json` do Script 1.

In [ ]:
# ── Persistência ─────────────────────────────────────────────────────────
TARGETS_VALIDOS = [t for t in TARGET_COLS if t in dfp.columns]

artefatos_df = {
    'dataset_preparado': dfp,
    'treino':            treino,
    'teste':             teste,
}
artefatos_meta = {
    'features':     FEATURES_SELECIONADAS,
    'targets':      TARGETS_VALIDOS,
    'kpis':         kpis_presentes,
    'cols_setor':   cols_setor,
    'cols_yoy':     cols_yoy,
    'params': {
        'limiar_nulo':   LIMIAR_NULO,
        'fator_winsor':  FATOR_WINSOR,
        'max_cols_yoy':  MAX_COLS_YOY,
        'clip_yoy':      CLIP_YOY,
        'corr_min':      CORR_MIN,
        'n_features_rfe':N_FEATURES_RFE,
        'frac_treino':   FRAC_TREINO,
    },
}

# Salva DataFrames como Parquet
for nome, df_art in artefatos_df.items():
    cam = PASTA_SAIDA / f'{nome}.parquet'
    df_art.to_parquet(cam, index=False)
    logger.info("Salvo: %s | %d × %d | %.0f KB",
                cam.name, *df_art.shape, cam.stat().st_size / 1024)

# Salva metadados como Pickle
for nome, obj in artefatos_meta.items():
    cam = PASTA_SAIDA / f'{nome}.pkl'
    with open(cam, 'wb') as f:
        pickle.dump(obj, f)
    logger.info("Salvo: %s", cam.name)

# Relatório de decisões em JSON (rastreabilidade acadêmica)
relatorio = {
    'dataset_input'       : str(cam_parquet),
    'dataset_empresas'    : int(dfp['CNPJ_CIA'].nunique()),
    'dataset_anos'        : sorted(dfp['ANO'].dropna().astype(int).unique().tolist()),
    'n_observacoes_total' : int(len(dfp)),
    'n_observacoes_treino': int(n_treino),
    'n_observacoes_teste' : int(n_teste),
    'n_features_candidatas': int(len(FEATURES_CANDIDATAS)),
    'n_features_selecionadas': int(len(FEATURES_SELECIONADAS)),
    'n_kpis'              : int(len(kpis_presentes)),
    'targets'             : TARGETS_VALIDOS,
    'kpis'                : kpis_presentes,
    'features'            : FEATURES_SELECIONADAS,
    'params'              : artefatos_meta['params'],
    'colunas_removidas_nulo': len(cols_excluir),
}
cam_rel = PASTA_SAIDA / 'relatorio_preparacao.json'
with open(cam_rel, 'w', encoding='utf-8') as f:
    json.dump(relatorio, f, indent=2, ensure_ascii=False, default=str)
logger.info("Relatório: %s", cam_rel)

# ── Resumo final ──────────────────────────────────────────────────────────
print("\n" + "═" * 65)
print("  RESUMO FINAL — Script 2 (cvm_preparacao_V2)")
print("═" * 65)
print(f"  Input        : {cam_parquet.name} ({n_total} obs brutas)")
print(f"  Output       : {len(dfp)} obs × {dfp.shape[1]} colunas (DFP anuais)")
print(f"  Empresas     : {dfp['CNPJ_CIA'].nunique()} / 25")
print(f"  Anos         : {sorted(dfp['ANO'].dropna().astype(int).unique())}")
print(f"  KPIs         : {len(kpis_presentes)} / {len(LISTA_KPIS)}")
print(f"  Features     : {len(FEATURES_SELECIONADAS)} selecionadas")
print(f"  Targets      : {TARGETS_VALIDOS}")
print(f"  Treino       : {n_treino} obs ({n_treino/len(dfp):.0%})")
print(f"  Teste        : {n_teste} obs ({n_teste/len(dfp):.0%})")
print(f"  Artefatos    : {PASTA_SAIDA}/")
print("═" * 65)
print("  ✅ Pronto para o Script 3 (03_cvm_modelagem.ipynb)")
print("═" * 65)
